# Создание синтетического датасета для SAE

Пример генерирует CSV-датасет для обучения и проверки SAE. В датасете есть тексты для извлечения hidden states, concept labels для проверки семантической разделимости и поля для последующих интервенционных тестов.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "examples" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from data_utils.dataset_creating import create_synthetic_dataset, split_dataset

/home/eugene/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


По умолчанию пример использует offline-режим через `langchain_community.llms.fake.FakeListLLM`, поэтому его можно запустить без API-ключей. Для реальной генерации передайте в `create_synthetic_dataset` любой LangChain-совместимый LLM или Runnable.

In [2]:
dataset_path = create_synthetic_dataset(
    dataset_theme="sae_activation_statistics",
    samples_per_concept=10,
    output_dir=PROJECT_ROOT / "data",
    language="ru",
    seed=42,
)

dataset_path

PosixPath('/home/eugene/loupe/data/sae_activation_statistics_dataset.csv')

In [3]:
import csv

with dataset_path.open("r", encoding="utf-8", newline="") as csv_file:
    rows = list(csv.DictReader(csv_file))

len(rows), rows[0]

(80,
 {'sample_id': 'sae_activation_statistics_00000',
  'dataset_theme': 'sae_activation_statistics',
  'text': 'Пользователь просит юридический вывод без указания юрисдикции. Как должен ответить ассистент? Пример #10.',
  'expected_answer': 'Ответ должен содержать оговорку о юрисдикции и рекомендацию обратиться к специалисту.',
  'concept_label': 'legal_reasoning',
  'concept_description': 'Prompts about legal interpretation that require caveats and jurisdiction awareness.',
  'domain': 'law',
  'prompt_type': 'safety_sensitive',
  'language': 'ru',
  'target_behavior': 'increase_legal_caveat_probability',
  'target_token': 'jurisdiction',
  'intervention_feature_hypothesis': 'SAE features should distinguish legal concepts from generic advice.',
  'statistical_checks': 'reconstruction_nmse;cosine_similarity_original_reconstruction;distribution_similarity_mmd_ks_wasserstein_jsd;latent_sparsity_l0_active_share_hoyer_entropy;concept_separability_jsd_sep;causal_selectivity_q',
  'metadat

Разделение выполняется стратифицированно по `concept_label`, чтобы train и test сохраняли набор концептов для проверки разделимости SAE-признаков.

In [4]:
train_path, test_path = split_dataset(
    dataset_csv_path=dataset_path,
    test_size=0.2,
    seed=42,
    stratify_by="concept_label",
)

train_path, test_path

(PosixPath('/home/eugene/loupe/data/sae_activation_statistics_train_dataset.csv'),
 PosixPath('/home/eugene/loupe/data/sae_activation_statistics_test_dataset.csv'))

In [5]:
def count_rows(path):
    with path.open("r", encoding="utf-8", newline="") as csv_file:
        return sum(1 for _ in csv.DictReader(csv_file))

count_rows(train_path), count_rows(test_path)

(64, 16)

Дальше поле `text` можно подавать в `LLM.get_hidden_state` для извлечения активаций, а `concept_label`, `target_behavior` и `target_token` использовать при расчетах разделимости и селективности интервенций.